# CASIE: verify DeBERTa GPU inference against the saved CPU predictions

This notebook performs inference and exact event-specific span-and-role scoring only. It does not train or push to GitHub. It checks all 736 test events and compares the GPU and CPU prediction sets event by event.

**Before running:** choose **Runtime → Change runtime type → GPU** (T4 if available). Run the cells in order. If dependency installation asks for a restart, restart the session and run from the top again.

You will upload two files when prompted:
1. `deberta_v3_base_run.zip` from your Downloads (about 744 MB).
2. `CASIE_CPU_test_predictions.jsonl` supplied alongside this notebook.

The public repository supplies the input bundle and code at a fixed commit. No Google Drive mount or personal path is required. GPU access is subject to Colab availability. Uploading the checkpoint here places it in your Colab runtime; it does not publish the weights on GitHub.

Historical notebook F1: **0.5932521087160262** (0.5933). Verified CPU F1: **0.5927835051546392** (0.5928). A GPU rerun can test whether the historical score is reproduced; a score match alone cannot prove the predictions equal the original missing GPU artifact.


In [ ]:
# 1. Install the recorded inference package versions. Do not import torch before this cell.
import subprocess
import sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.11.0", "transformers==5.21.0",
    "tokenizers==0.23.3", "sentencepiece==0.2.2",
])
print("Dependencies installed. Restart the session if Colab asks you to.")


In [ ]:
# 2. Fetch the exact repository revision used for the CPU run.
import csv
import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import time
import zipfile

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK = Path("/content/casie_deberta_verification")
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "repository"
COMMIT = "e563f31d273e7f03c6b566d211c6f67d340f095e"
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--quiet",
        "https://github.com/RanimAlyaseen/casie-event-argument-extraction.git", str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", COMMIT])
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == COMMIT
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Use an unmodified pinned checkout."

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

EXPECTED_SOURCES = {
    "casie_transformer.py": "6a6016764dc3b3bbec717c7ad5a9ca9eb2cf6947f3ecca7b41973ba2f7984759",
    "casie_eval.py": "9bf9e79ca3fee0e48373c4ad1844dacd09ad37755942bd66d33de286a870961f",
}
for name, expected in EXPECTED_SOURCES.items():
    assert sha256(REPO / "team_code" / name) == expected, f"Unexpected source: {name}"
print("Verified pinned code:", COMMIT)


## Upload the checkpoint and the CPU predictions

Select both files in the upload dialog. The checkpoint ZIP is large, so wait for the upload to finish before proceeding. You can alternatively upload them through Colab's Files panel into `/content/casie_deberta_verification/uploads/`; the next cell skips files already there.

The model stays in the runtime. The final downloadable results archive excludes the model weights and dataset.


In [ ]:
# 3. Upload inputs; rerunning this cell reuses files already uploaded.
from google.colab import files
UPLOADS = WORK / "uploads"
UPLOADS.mkdir(exist_ok=True)
CHECKPOINT_ZIP = UPLOADS / "deberta_v3_base_run.zip"
CPU_FILE = UPLOADS / "CASIE_CPU_test_predictions.jsonl"
if not CHECKPOINT_ZIP.is_file() or not CPU_FILE.is_file():
    previous_directory = Path.cwd()
    try:
        os.chdir(UPLOADS)
        uploaded = files.upload()
        del uploaded
    finally:
        os.chdir(previous_directory)
    gc.collect()
for path in (CHECKPOINT_ZIP, CPU_FILE):
    assert path.is_file(), f"Missing upload: {path.name}"
assert sha256(CPU_FILE) == "1d170c66909d1704fc0e8a3eca500beaebbcd4160bb6b24f578aca3b18fca3db", "This is not the verified CPU prediction file."
print("Uploads found; CPU predictions checksum verified.")


In [ ]:
# 4. Extract only checkpoint files into this temporary Colab workspace.
CHECKPOINT = WORK / "checkpoint" / "deberta_v3_base"
CHECKPOINT.mkdir(parents=True, exist_ok=True)
MEMBERS = ["config.json", "tokenizer.json", "tokenizer_config.json",
           "model.safetensors", "training_config.json"]
with zipfile.ZipFile(CHECKPOINT_ZIP) as archive:
    for name in MEMBERS:
        member = "outputs/deberta_v3_base/" + name
        info = archive.getinfo(member)
        destination = CHECKPOINT / name  # fixed whitelist; no arbitrary ZIP paths
        if not destination.exists():
            with archive.open(info) as source, destination.open("xb") as target:
                shutil.copyfileobj(source, target)
        assert destination.stat().st_size == info.file_size, f"Incomplete extraction: {name}"
WEIGHTS_SHA = sha256(CHECKPOINT / "model.safetensors")
assert WEIGHTS_SHA == "ace9511fa25e3056e2044ff5839f53aa4441e0bc9a47ae220f290a5fef9058fe", "Checkpoint weights differ from the CPU run."
print("Restored and verified checkpoint:", CHECKPOINT)


In [ ]:
# 5. Read the test member directly from the bundled ZIP; preserve event IDs and document text.
sys.path.insert(0, str(REPO / "team_code"))
import casie_eval as ce
records, documents = [], {}
test_digest = hashlib.sha256()
with zipfile.ZipFile(REPO / "data/TeamUltron_Project2_Inputs.zip") as archive:
    expected_test_sha = json.loads(archive.read("input_manifest.json"))["files"]["dataset/CASIE_test.jsonl"]
    with archive.open("dataset/CASIE_test.jsonl") as source:
        for line in source:
            test_digest.update(line)
            if not line.strip():
                continue
            raw = json.loads(line)
            document_id = raw["document_id"]
            if document_id in documents:
                assert documents[document_id] == raw["tokens"]
            else:
                documents[document_id] = raw["tokens"]
            record = {key: raw[key] for key in
                      ("document_id", "event_uid", "event_subtype", "trigger", "arguments")}
            record["tokens"] = documents[document_id]
            records.append(record)
TEST_SHA = test_digest.hexdigest()
assert TEST_SHA == expected_test_sha == "ed9c585a54660d14decdd49efd25dfb33e07c44f150614fb136c2cbfbdbc1c50"
EVENT_IDS = {record["event_uid"] for record in records}
assert len(records) == len(EVENT_IDS) == 736
assert sum(len(ce.gold_spans(record)) for record in records) == 2109
cpu_preds = ce.load_predictions(str(CPU_FILE))
assert set(cpu_preds) == EVENT_IDS, "CPU file has missing or unknown event IDs."
cpu_result = ce.evaluate(records, cpu_preds, n_examples=0)
assert abs(cpu_result["overall"]["arg_c_f1"] - 0.5927835051546392) < 1e-12
print("Verified 736 events, 100 documents, 2109 gold arguments.")
print("CPU:", cpu_result["overall"], "warnings:", cpu_result["warnings"])


## Run GPU inference

This uses the repository's `load_trained()` and `predict_file()` exactly as the executed notebook did: maximum length 512, predictions sorted by encoded length, **batch size 32**, and CUDA float16 autocast inside `predict()`. It does not call training code.

If CUDA is unavailable, switch the runtime to GPU. If batch 32 causes GPU out-of-memory, the cell stops; use a runtime with more GPU memory for a matched comparison. A smaller batch is a separate configuration and should be labeled accordingly.


In [ ]:
# 6. Check the runtime and load the verified fine-tuned checkpoint offline.
from importlib.metadata import version
import platform
import torch
assert torch.__version__.split("+")[0] == "2.11.0", "Restart after installation, then rerun from the top."
assert torch.cuda.is_available(), "Enable a GPU under Runtime > Change runtime type, then reconnect."
os.environ["HF_HUB_OFFLINE"] = "1"
import casie_transformer as ctr
runtime = {
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ("torch", "transformers", "tokenizers", "sentencepiece")},
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "batch_size": 32,
    "autocast": "CUDA float16 (repository predict implementation)",
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "repository_commit": COMMIT,
    "weights_sha256": WEIGHTS_SHA,
    "test_sha256": TEST_SHA,
    "source_sha256": EXPECTED_SOURCES,
    "cpu_predictions_sha256": sha256(CPU_FILE),
}
model, tokenizer, encoder = ctr.load_trained(str(CHECKPOINT), max_len=512)
assert next(model.parameters()).device.type == "cuda"
assert len(tokenizer) == model.get_input_embeddings().num_embeddings == 128003
model.eval()
print(json.dumps(runtime, indent=2))


In [ ]:
# 7. Predict all 736 events using the same entry point as the original notebook.
OUT = WORK / ("results_" + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()))
OUT.mkdir(exist_ok=False)
GPU_FILE = OUT / "gpu_test_predictions.jsonl"
torch.cuda.synchronize()
started = time.perf_counter()
try:
    gpu_preds = ctr.predict_file(model, tokenizer, encoder, records, str(GPU_FILE))
except torch.cuda.OutOfMemoryError as error:
    raise RuntimeError("GPU batch 32 ran out of memory. No smaller-batch fallback was applied.") from error
torch.cuda.synchronize()
runtime["inference_seconds"] = time.perf_counter() - started
assert set(gpu_preds) == EVENT_IDS
assert ce.load_predictions(str(GPU_FILE)) == gpu_preds
gpu_result = ce.evaluate(records, gpu_preds, n_examples=0)
print("GPU:", gpu_result["overall"])
print("Scorer warnings:", gpu_result["warnings"])
print("Inference seconds:", round(runtime["inference_seconds"], 2))


In [ ]:
# 8. Compare exact per-event argument sets, including whether changed spans are gold.
def compare_predictions(gold_records, cpu, gpu):
    expected = {record["event_uid"] for record in gold_records}
    assert set(cpu) == set(gpu) == expected, "Comparison populations differ."
    changes = []
    for record in gold_records:
        uid = record["event_uid"]
        gold = ce.gold_spans(record)
        old, new = cpu[uid], gpu[uid]
        if old == new:
            continue
        changes.append({
            "event_uid": uid,
            "document_id": record["document_id"],
            "event_subtype": record["event_subtype"],
            "cpu_correct": len(old & gold),
            "gpu_correct": len(new & gold),
            "net_correct_difference": len(new & gold) - len(old & gold),
            "cpu_only": [{"start": s, "end": e, "role": role,
                          "text": " ".join(record["tokens"][s:e + 1]),
                          "is_gold": (s, e, role) in gold} for s, e, role in sorted(old - new)],
            "gpu_only": [{"start": s, "end": e, "role": role,
                          "text": " ".join(record["tokens"][s:e + 1]),
                          "is_gold": (s, e, role) in gold} for s, e, role in sorted(new - old)],
        })
    return changes

changes = compare_predictions(records, cpu_preds, gpu_preds)
HISTORICAL_F1 = 0.5932521087160262
comparison = {
    "cpu": cpu_result["overall"],
    "gpu": gpu_result["overall"],
    "cpu_scorer_warnings": cpu_result["warnings"],
    "gpu_scorer_warnings": gpu_result["warnings"],
    "historical_notebook_f1": HISTORICAL_F1,
    "gpu_matches_historical_f1": abs(gpu_result["overall"]["arg_c_f1"] - HISTORICAL_F1) < 1e-12,
    "gpu_minus_cpu_f1": gpu_result["overall"]["arg_c_f1"] - cpu_result["overall"]["arg_c_f1"],
    "gpu_minus_historical_f1": gpu_result["overall"]["arg_c_f1"] - HISTORICAL_F1,
    "changed_events": len(changes),
    "identical_events": len(records) - len(changes),
    "net_correct_difference": sum(row["net_correct_difference"] for row in changes),
    "interpretation": "Backend, arithmetic precision, and batch size differ together. This comparison cannot isolate their individual effects. The original historical per-event GPU file is unavailable.",
    "runtime": runtime,
}
print(json.dumps(comparison, indent=2))
print("First changed events:", json.dumps(changes[:10], indent=2))
if comparison["gpu_matches_historical_f1"]:
    print("GPU F1 matches the historical score. Inspect the event differences before attributing a cause.")
else:
    print("GPU F1 still differs from the historical score. Preserve all artifacts; do not replace the published result.")


In [ ]:
# 9. Save and download predictions, scores, and the exact differences. No weights/data are included.
(OUT / "comparison_report.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
(OUT / "event_differences.json").write_text(json.dumps(changes, indent=2), encoding="utf-8")
ce.save_predictions(str(OUT / "cpu_test_predictions.jsonl"), cpu_preds)
with (OUT / "event_differences.csv").open("w", encoding="utf-8", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["event_uid", "document_id", "event_subtype",
        "cpu_correct", "gpu_correct", "net_correct_difference", "cpu_only", "gpu_only"])
    writer.writeheader()
    for row in changes:
        writer.writerow({**row, "cpu_only": json.dumps(row["cpu_only"]),
                         "gpu_only": json.dumps(row["gpu_only"])})
DOWNLOAD = WORK / (OUT.name + "_CASIE_verification.zip")
allowed = ["gpu_test_predictions.jsonl", "cpu_test_predictions.jsonl",
           "comparison_report.json", "event_differences.json", "event_differences.csv"]
with zipfile.ZipFile(DOWNLOAD, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in allowed:
        archive.write(OUT / name, arcname=name)
print("Download this archive and send it back for review:", DOWNLOAD.name)
files.download(str(DOWNLOAD))


## What to send back

Download the verification ZIP from the last cell and share it for review. It contains both prediction files, both scores, runtime information, and exact changed event/span details. It excludes the checkpoint weights and dataset.

If a cell fails, send the complete error text and the cell number. Nothing in this notebook modifies the public repository or the historical notebook result. GPU execution remains unverified until you run it in Colab.

Colab reference: https://research.google.com/colaboratory/faq.html
